# Interpolation validation

Checks whether `interp_data` interpolates missing points close to their true values. Known points are removed, interpolated back in, and scored with MAE/RMSE.

In [ ]:
import sys
from pathlib import Path
from itertools import groupby
import contextlib, io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

# make read_data / plotter importable from here
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent / "plotter"))
from read_data import load_subject, interp_data   # module under test

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
import os

# DATA_DIR is the PAMAP2 folder. Defaults to the repo-relative path;
# set the PAMAP2_DIR environment variable if your copy is elsewhere.
_candidates = [os.environ.get("PAMAP2_DIR"), "../PAMAP2_Dataset/Protocol"]
DATA_DIR = next((str(Path(p)) + "/" for p in _candidates
                 if p and (Path(p) / "subject101.dat").exists()),
                "../data/PAMAP2_Dataset/Protocol/")
print("Using data from:", DATA_DIR)

SUBJECT     = "101"
COL         = "hand_gyro_x"   # one wrist-gyroscope axis
ACTIVITY_ID = 5               # running: fast and oscillating, hardest case to interpolate
SAMPLE_RATE = 100             # Hz -> 100 rows = 1 second

df = load_subject(SUBJECT, folder_path=DATA_DIR)

## Gap-free block

A gap-free run gives a reference for comparing the interpolator against the actual data. Running wrist-gyro is the hardest case. It is fast and oscillating, so a straight-line fit deviates most.

In [ ]:
def longest_gap_free_segment(df, col, activity_id, want=1500):
    """Return the indices of a gap-free, index-contiguous run of `col` during one activity."""
    rows = df[df["activity_id"] == activity_id]
    idx  = rows.index.to_numpy()
    ok   = rows[col].notna().to_numpy()
    blocks, s = [], 0
    for i in range(1, len(idx) + 1):
        broken = (i == len(idx)) or (not ok[i]) or (idx[i] - idx[i-1] != 1)
        if broken:
            if s < len(idx) and ok[s]:      # guard: s may point past the end
                blocks.append(idx[s:i])
            s = i if (i < len(idx) and ok[i]) else i + 1   # skip the missing (NaN) row
    blocks = [b for b in blocks if len(b) >= want]
    return max(blocks, key=len) if blocks else None

block = longest_gap_free_segment(df, COL, ACTIVITY_ID)[:1500]   # 15 s of gap-free signal
truth = df.loc[block, COL].copy()                          # ground truth, hidden later
print(f"Clean block: rows {block[0]}..{block[-1]}  ({len(block)} samples = {len(block)/SAMPLE_RATE:.0f}s)")

## Actual gap sizes

Actual gaps are the runs of consecutive missing samples that occur in the recording. This is how big they get in this column, which is what interpolation has to handle.

In [ ]:
def gap_lengths(mask):
    idx = mask[mask].index.values
    return np.array([len(list(g)) for _, g in groupby(enumerate(idx), lambda x: x[0]-x[1])])

actual_gaps = gap_lengths(df[COL].isnull())
print(f"{COL}: {len(actual_gaps)} actual gaps")
print(f"  median : {np.median(actual_gaps):.0f} samples ({np.median(actual_gaps)/SAMPLE_RATE*1000:.0f} ms)")
print(f"  90th % : {np.percentile(actual_gaps,90):.0f} samples")
print(f"  max    : {actual_gaps.max()} samples ({actual_gaps.max()/SAMPLE_RATE*1000:.0f} ms)")

Almost every actual gap is a single dropped sample, never more than ~0.1 s.

## Hold-out test

Delete known values to make gaps of a set size, interpolate them with `interp_data`, then score the interpolation against the deleted values (MAE/RMSE). Sizes run from the actual range up to large gaps to find where the interpolation becomes inaccurate.

In [ ]:
def _silent(fn, *a, **k):
    with contextlib.redirect_stdout(io.StringIO()):   # interp_data prints progress; suppress it
        return fn(*a, **k)

def mask_and_recover(df, col, block, truth, gap_size, n_gaps=8, pad=20, t_window=3):
    """Hide n_gaps stretches of length gap_size inside `block`, interpolate, return (y_true, y_pred)."""
    df_masked = df.copy()
    stride = max(gap_size + pad, (len(block) - 2*pad) // n_gaps)  # keep gaps from overlapping
    starts = pad + np.arange(n_gaps) * stride
    starts = starts[starts + gap_size <= len(block) - pad]        # drop any that run off the end
    masked = [block[st:st+gap_size] for st in starts]
    for gi in masked:
        df_masked.loc[gi, col] = np.nan
    df_filled = _silent(interp_data, df_masked, t_window, [col])
    y_true = np.concatenate([truth.loc[gi].to_numpy()        for gi in masked])
    y_pred = np.concatenate([df_filled.loc[gi, col].to_numpy() for gi in masked])
    return y_true, y_pred

In [ ]:
gap_sizes = [1, 2, 3, 5, 10, 20, 50, 100, 150, 250]   # all < 3s window, so all get interpolated
rows = []
for gs in gap_sizes:
    yt, yp = mask_and_recover(df, COL, block, truth, gs)
    keep = ~np.isnan(yp)
    rows.append({
        "gap_samples": gs,
        "gap_ms":      gs / SAMPLE_RATE * 1000,
        "MAE":         mean_absolute_error(yt[keep], yp[keep]),
        "RMSE":        np.sqrt(mean_squared_error(yt[keep], yp[keep])),
        "pct_filled":  100 * keep.mean(),
    })
results = pd.DataFrame(rows)
signal_scale = truth.std()
results["MAE_%_of_signal"] = 100 * results["MAE"] / signal_scale
print(f"signal scale (std of true values): {signal_scale:.3f} rad/s\n")
results.round(3)

`MAE_%_of_signal` is the error as a fraction of the signal's standard deviation. It is under ~10% for short gaps and increases with gap length.

## Small vs. large gap

A realistic 2-sample gap next to a large 1.5 s one.

In [ ]:
def fill_one(df, col, gi, t_window=3):
    d = df.copy(); d.loc[gi, col] = np.nan
    return _silent(interp_data, d, t_window, [col])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, demo, title in [(axes[0], 2, "small gap (2 samples)"),
                        (axes[1], 150, "large gap (150 samples = 1.5s)")]:
    st  = 600
    gi  = block[st:st+demo]
    fill = fill_one(df, COL, gi)
    ctx = max(60, demo)
    win = block[st-ctx: st+demo+ctx]
    t   = np.arange(len(win)) / SAMPLE_RATE
    ax.plot(t, truth.loc[win].to_numpy(), color="0.45", lw=1.3, label="true (hidden)")
    ax.plot(t, fill.loc[win, COL].to_numpy(), color="C1", lw=1.3, ls="--", label="interpolated")
    ax.axvspan(ctx/SAMPLE_RATE, (ctx+demo)/SAMPLE_RATE, color="C3", alpha=0.12)
    ax.set_title(title); ax.set_xlabel("time (s)"); ax.legend(loc="upper right")
axes[0].set_ylabel(f"{COL} (rad/s)")
fig.suptitle(f"interp_data across a masked gap — running, subject {SUBJECT}")
fig.tight_layout(); plt.show()

On the 2-sample gap the interpolation matches the true signal. On the 1.5 s gap it is a straight line and misses the oscillation.

## Error vs. gap length

MAE against gap length, with the real-gap range marked.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(results["gap_samples"], results["MAE"], "o-", color="C0", label="interpolation MAE")
plt.axhline(signal_scale, color="0.5", ls=":",
            label="signal std (≈ error of just guessing the average)")
plt.axvspan(0, actual_gaps.max(), color="C2", alpha=0.15,
            label=f"range of actual gaps (≤ {actual_gaps.max()} samples)")
plt.xlabel("gap length (samples;  100 = 1 second)")
plt.ylabel("MAE (rad/s)")
plt.title("Linear interpolation: accurate for small gaps, degrades for large ones")
plt.legend(); plt.tight_layout(); plt.show()

## Heart rate

Heart rate is the low-frequency case. It is sampled at about 9 Hz while the IMU runs at 100 Hz, so it is NaN on most rows by design, not from dropouts. `interp_data` skips it, and there is no gap-free block to mask, so the check runs on the measured readings: hide some, interpolate each from its neighbors by timestamp, and score against what was hidden.

In [ ]:
# Heart rate is sampled at ~9 Hz, so it is NaN on most 100 Hz rows by design, not from dropouts.
# interp_data skips it and there is no gap-free block, so validate on the measured readings by timestamp.
hr = df[df["activity_id"] == ACTIVITY_ID][["timestamp", "heart_rate"]].dropna().reset_index(drop=True)
t_hr, y_hr = hr["timestamp"].to_numpy(), hr["heart_rate"].to_numpy()
hr_scale = y_hr.std()

def hr_recover(gap_size, n_gaps=200, pad=5):
    # hide gap_size consecutive readings at spread-out spots, interpolate each from the readings on either side
    yt, yp = [], []
    stride = max(gap_size + 2, (len(y_hr) - 2*pad) // n_gaps)
    for st in range(pad, len(y_hr) - gap_size - pad, stride):
        lo, hi = st - 1, st + gap_size
        for j in range(st, st + gap_size):
            yt.append(y_hr[j])
            yp.append(np.interp(t_hr[j], [t_hr[lo], t_hr[hi]], [y_hr[lo], y_hr[hi]]))
    return np.array(yt), np.array(yp)

spacing = np.median(np.diff(t_hr))
rows = []
for gs in [1, 2, 3, 5, 8, 12]:
    yt, yp = hr_recover(gs)
    rows.append({
        "hidden_readings": gs,
        "gap_s":           round(gs * spacing, 2),
        "MAE_bpm":         mean_absolute_error(yt, yp),
        "RMSE_bpm":        np.sqrt(mean_squared_error(yt, yp)),
        "MAE_%_of_signal": 100 * mean_absolute_error(yt, yp) / hr_scale,
    })
results_hr = pd.DataFrame(rows)
print(f"running HR: {len(y_hr)} readings, {y_hr.min():.0f}-{y_hr.max():.0f} bpm, std {hr_scale:.1f} bpm\n")
results_hr.round(3)

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(results["gap_ms"]/1000, results["MAE_%_of_signal"], "o-", label="hand_gyro_x (100 Hz)")
plt.plot(results_hr["gap_s"], results_hr["MAE_%_of_signal"], "s-", label="heart_rate (~9 Hz)")
plt.xlabel("gap duration (s)")
plt.ylabel("MAE (% of signal std)")
plt.title("MAE vs gap duration: heart rate vs gyro")
plt.legend(); plt.tight_layout(); plt.show()

Heart rate interpolates very accurately. The error stays below 1% of the signal even for gaps of about a second, because it drifts slowly relative to its sampling, so a straight line between readings deviates little. This is the opposite of the gyro, where fast oscillation makes long gaps hard.

## Conclusion

The actual gaps in this data are all short, and linear interpolation recovers them accurately, so it is a sound way to fill the gaps in this signal. The interpolation only becomes unreliable on gaps far longer than any that occur here. It should be noted `interp_data` interpolates gaps up to 3 s by default. That is much larger than needed, and reaches into the range where the interpolation is poor, so a shorter limit would be safer.